In [ ]:
"""
CNN SPORTS CLASSIFIER — COMPLETE PIPELINE
============================================
Runs top to bottom as a single script (or paste each "# %%" block as its
own Jupyter cell). Uses transfer learning (EfficientNetB0) by default.

Alternative model options (ResNet50 backbone, from-scratch CNN) and the
ensembling section are included but wrapped in triple-quoted blocks so
they don't run automatically — see the comments at each to activate them.

SECTIONS:
  1. Imports
  2. Unzip dataset
  3. Device setup
  4. Data: transforms, datasets, loaders (with augmentation)
  5. Sanity check: inspect one batch
  6. Model — EfficientNetB0 transfer learning (+ alternatives, inactive)
  7. Loss, optimizer, scheduler
  8. Forward-pass sanity check
  9. Training loop function
 10. Run initial training
 11. Optional: Phase 2 fine-tuning (gradual unfreezing)
 12. Test set evaluation
 13. Save model
 14. Visual prediction test
 15. Sample image grid
 16. Class distribution chart
 17. Feature map visualization
 18. Predictions grid (correct + incorrect)
 19. Confusion matrix
 20. Grad-CAM
 21. Test-time augmentation (TTA)
 22. Ensembling (inactive — needs 2+ separately trained models)
"""

# %% ================================================================
# 1. IMPORTS
# ====================================================================
import os
import zipfile
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import confusion_matrix
# if sklearn isn't installed, run this once in its own cell first:
# !pip install scikit-learn


# %% ================================================================
# 2. UNZIP DATASET
# ====================================================================
zip_path = 'archive.zip'
extract_path = 'image_dataset/'

if not os.path.exists(os.path.join(extract_path, 'train')):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Unzipping complete!")
else:
    print("Dataset already extracted — skipping unzip.")


# %% ================================================================
# 3. DEVICE SETUP
# ====================================================================
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Using device: {device}")


# %% ================================================================
# 4. DATA — transforms, datasets, loaders (augmentation on train only)
# ====================================================================
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dir = os.path.join(extract_path, 'train')
valid_dir = os.path.join(extract_path, 'valid')
test_dir = os.path.join(extract_path, 'test')

train_dataset = torchvision.datasets.ImageFolder(root=train_dir, transform=train_transform)
valid_dataset = torchvision.datasets.ImageFolder(root=valid_dir, transform=eval_transform)
test_dataset = torchvision.datasets.ImageFolder(root=test_dir, transform=eval_transform)

BATCH_SIZE = 32
trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
testloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

classes = train_dataset.classes
num_classes = len(classes)

print(f"Classes found: {num_classes}")
print(f"Train samples: {len(train_dataset)}")
print(f"Valid samples: {len(valid_dataset)}")
print(f"Test samples:  {len(test_dataset)}")


# %% ================================================================
# 5. SANITY CHECK — inspect one batch before building the model
# ====================================================================
images, labels = next(iter(trainloader))
print(f"Batch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")
print(f"Label range: {labels.min().item()} to {labels.max().item()}")


# %% ================================================================
# 6. MODEL — EfficientNetB0 transfer learning (ACTIVE by default)
# ====================================================================
weights_eff = EfficientNet_B0_Weights.DEFAULT
net = efficientnet_b0(weights=weights_eff)

for param in net.features.parameters():   # freeze the pretrained backbone
    param.requires_grad = False

net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
net = net.to(device)

print(net.__class__.__name__, "loaded with", sum(p.numel() for p in net.parameters() if p.requires_grad), "trainable params")

# ---- ALTERNATIVE — ResNet50 backbone instead of EfficientNetB0 ----
# To activate: delete the triple quotes below, and comment out the
# EfficientNetB0 block above (lines from weights_eff = ... to net = net.to(device))
"""
weights_res = ResNet50_Weights.DEFAULT
net = resnet50(weights=weights_res)
for param in net.parameters():
    param.requires_grad = False
net.fc = nn.Linear(net.fc.in_features, num_classes)
net = net.to(device)
"""

# ---- ALTERNATIVE — from-scratch CNN instead of transfer learning ----
# To activate: delete the triple quotes below, and comment out the
# EfficientNetB0 block above
"""
class Net(nn.Module):
    def __init__(self, num_classes):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            dummy = self.pool(F.relu(self.bn1(self.conv1(dummy))))
            dummy = self.pool(F.relu(self.bn2(self.conv2(dummy))))
            dummy = self.pool(F.relu(self.bn3(self.conv3(dummy))))
            flat_size = dummy.numel()
        self.fc1 = nn.Linear(flat_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.flatten(1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

net = Net(num_classes=num_classes).to(device)
"""


# %% ================================================================
# 7. LOSS, OPTIMIZER, SCHEDULER
# ====================================================================
USE_CLASS_WEIGHTS = False   # flip True if section 16 shows heavy imbalance

if USE_CLASS_WEIGHTS:
    class_counts = Counter(train_dataset.targets)
    counts = torch.tensor([class_counts[i] for i in range(num_classes)], dtype=torch.float)
    weights = (1.0 / counts)
    weights = (weights / weights.sum() * num_classes).to(device)
else:
    weights = None

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)


# %% ================================================================
# 8. FORWARD-PASS SANITY CHECK
# ====================================================================
net.eval()
with torch.no_grad():
    sample_images, sample_labels = next(iter(trainloader))
    sample_images = sample_images.to(device)
    sample_outputs = net(sample_images)
    print(f"Output shape: {sample_outputs.shape}")
    assert sample_outputs.shape[1] == num_classes, "Output size doesn't match num_classes!"
    print("Forward pass OK — shapes line up correctly.")
net.train()


# %% ================================================================
# 9. TRAINING LOOP FUNCTION
# Reused for both initial training and the optional fine-tuning phase
# ====================================================================
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def train_and_validate(net, epochs, criterion, optimizer, scheduler,
                        trainloader, validloader, device,
                        use_mixup=False, mixup_alpha=0.2,
                        early_stop_patience=5, checkpoint_path='best_model.pth',
                        best_val_acc=0.0):
    print_every = max(1, len(trainloader) // 5)
    epochs_without_improvement = 0

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            if use_mixup:
                inputs, labels_a, labels_b, lam = mixup_data(inputs, labels, mixup_alpha)
                outputs = net(inputs)
                loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
            else:
                outputs = net(inputs)
                loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % print_every == print_every - 1:
                print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / print_every))
                running_loss = 0.0

        # ---- Train accuracy ----
        net.eval()
        train_correct, train_total = 0, 0
        with torch.no_grad():
            for data in trainloader:
                images, labels = data
                images, labels = images.to(device), labels.to(device)
                outputs = net(images)
                _, predicted = torch.max(outputs, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()
        train_acc = 100 * train_correct / train_total
        print(f'Train accuracy: {train_acc:.2f}%')

        # ---- Validation accuracy ----
        net.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for data in validloader:
                images, labels = data
                images, labels = images.to(device), labels.to(device)
                outputs = net(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        val_acc = 100 * val_correct / val_total
        print(f'>>> Epoch {epoch+1} validation accuracy: {val_acc:.2f}%')

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            torch.save(net.state_dict(), checkpoint_path)
            print(f'  New best model saved ({val_acc:.2f}%)')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= early_stop_patience:
                print("Early stopping triggered")
                break

    print(f'Finished — best validation accuracy so far: {best_val_acc:.2f}%')
    return best_val_acc


# %% ================================================================
# 10. RUN INITIAL TRAINING
# ====================================================================
EPOCHS = 30
USE_MIXUP = False        # flip True to enable mixup augmentation
MIXUP_ALPHA = 0.2
EARLY_STOP_PATIENCE = 5

best_val_acc = train_and_validate(
    net, EPOCHS, criterion, optimizer, scheduler, trainloader, validloader, device,
    use_mixup=USE_MIXUP, mixup_alpha=MIXUP_ALPHA,
    early_stop_patience=EARLY_STOP_PATIENCE, checkpoint_path='best_model.pth'
)


# %% ================================================================
# 11. OPTIONAL — PHASE 2 FINE-TUNING (gradual unfreezing)
# Only meaningful if using the EfficientNetB0 model from section 6.
# Comment this whole section out if you'd rather stop after section 10.
# ====================================================================
net.load_state_dict(torch.load('best_model.pth'))

for param in net.features[-2:].parameters():   # unfreeze last two feature blocks
    param.requires_grad = True

optimizer = optim.Adam(net.parameters(), lr=1e-5)   # small LR — don't wreck pretrained weights
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

FINE_TUNE_EPOCHS = 10
best_val_acc = train_and_validate(
    net, FINE_TUNE_EPOCHS, criterion, optimizer, scheduler, trainloader, validloader, device,
    checkpoint_path='best_model.pth', best_val_acc=best_val_acc
)


# %% ================================================================
# 12. TEST SET EVALUATION
# ====================================================================
net.load_state_dict(torch.load('best_model.pth'))
net.eval()
correct, total = 0, 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test accuracy: {100 * correct / total:.2f}%')


# %% ================================================================
# 13. SAVE MODEL (already checkpointed as best_model.pth during training —
# this just saves a copy under a separate final name if you want one)
# ====================================================================
torch.save(net.state_dict(), 'final_sports_cnn.pth')
print("Model saved as final_sports_cnn.pth")


# %% ================================================================
# 14. VISUAL PREDICTION TEST
# ====================================================================
def show_prediction(net, dataset, idx, classes, device):
    net.eval()
    image, true_label = dataset[idx]
    with torch.no_grad():
        output = net(image.unsqueeze(0).to(device))
        _, predicted = torch.max(output, 1)

    img_display = image.permute(1, 2, 0).numpy() * 0.5 + 0.5
    img_display = img_display.clip(0, 1)

    plt.imshow(img_display)
    correct = predicted.item() == true_label
    plt.title(f"True: {classes[true_label]} | Predicted: {classes[predicted.item()]}",
              color='green' if correct else 'red')
    plt.axis('off')
    plt.show()

for _ in range(3):
    idx = random.randint(0, len(test_dataset) - 1)
    show_prediction(net, test_dataset, idx, classes, device)


# %% ================================================================
# 15. SAMPLE IMAGE GRID
# ====================================================================
def show_sample_grid(dataset, classes, n=9):
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    indices = random.sample(range(len(dataset)), n)
    for ax, idx in zip(axes.flat, indices):
        image, label = dataset[idx]
        img = image.permute(1, 2, 0).numpy() * 0.5 + 0.5
        ax.imshow(img.clip(0, 1))
        ax.set_title(classes[label], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_grid(train_dataset, classes)


# %% ================================================================
# 16. CLASS DISTRIBUTION CHART
# ====================================================================
class_counts = Counter(train_dataset.targets)
counts_sorted = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
names = [classes[i] for i, _ in counts_sorted]
values = [c for _, c in counts_sorted]

plt.figure(figsize=(14, 5))
plt.bar(range(len(values)), values)
plt.xticks(range(len(values)), names, rotation=90, fontsize=6)
plt.ylabel('Training images')
plt.title('Images per class')
plt.tight_layout()
plt.show()

print(f"Min: {min(values)}, Max: {max(values)} — imbalance ratio: {max(values)/min(values):.1f}x")


# %% ================================================================
# 17. FEATURE MAP VISUALIZATION
# ====================================================================
def visualize_feature_maps(net, image_tensor, layer, num_filters=6):
    net.eval()
    activation = {}
    def hook(module, input, output):
        activation['feat'] = output.detach()

    handle = layer.register_forward_hook(hook)
    with torch.no_grad():
        net(image_tensor.unsqueeze(0).to(device))
    handle.remove()

    feat_maps = activation['feat'].squeeze(0).cpu()
    num_filters = min(num_filters, feat_maps.shape[0])

    fig, axes = plt.subplots(1, num_filters, figsize=(15, 3))
    for i in range(num_filters):
        axes[i].imshow(feat_maps[i], cmap='viridis')
        axes[i].set_title(f'Filter {i}')
        axes[i].axis('off')
    plt.show()

image, label = test_dataset[0]
img_display = (image.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
plt.imshow(img_display)
plt.title(f"Original: {classes[label]}")
plt.axis('off')
plt.show()

# net.features[0] = first block, net.features[-1] = last block (EfficientNetB0)
# If you switched to the from-scratch CNN option, use net.conv1 / net.conv2 instead
visualize_feature_maps(net, image, net.features[0], num_filters=6)


# %% ================================================================
# 18. PREDICTIONS GRID (correct + incorrect)
# ====================================================================
def show_predictions_grid(net, dataset, classes, n=9, only_wrong=False):
    net.eval()
    indices = list(range(len(dataset)))
    random.shuffle(indices)

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes = axes.flat
    shown = 0

    with torch.no_grad():
        for idx in indices:
            if shown >= n:
                break
            image, true_label = dataset[idx]
            output = net(image.unsqueeze(0).to(device))
            probs = F.softmax(output, dim=1)
            conf, predicted = torch.max(probs, 1)
            predicted, conf = predicted.item(), conf.item()

            is_correct = predicted == true_label
            if only_wrong and is_correct:
                continue

            img = (image.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
            axes[shown].imshow(img)
            axes[shown].set_title(
                f"True: {classes[true_label]}\nPred: {classes[predicted]} ({conf*100:.0f}%)",
                fontsize=9, color='green' if is_correct else 'red'
            )
            axes[shown].axis('off')
            shown += 1

    plt.tight_layout()
    plt.show()

show_predictions_grid(net, test_dataset, classes, n=9)
show_predictions_grid(net, test_dataset, classes, n=9, only_wrong=True)


# %% ================================================================
# 19. CONFUSION MATRIX + MOST-CONFUSED PAIRS
# ====================================================================
all_preds, all_labels = [], []
net.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

cm_copy = cm.copy()
np.fill_diagonal(cm_copy, 0)
flat_indices = np.argsort(cm_copy, axis=None)[::-1][:10]
top_pairs = np.unravel_index(flat_indices, cm_copy.shape)

print("Most confused class pairs (true -> predicted, count):")
for true_idx, pred_idx in zip(*top_pairs):
    if cm_copy[true_idx, pred_idx] > 0:
        print(f"  {classes[true_idx]} -> {classes[pred_idx]}: {cm_copy[true_idx, pred_idx]}")


# %% ================================================================
# 20. GRAD-CAM (where the model is "looking")
# ====================================================================
def grad_cam(net, image_tensor, target_layer, class_idx=None):
    net.eval()
    activations, gradients = {}, {}

    def forward_hook(module, input, output):
        activations['value'] = output
    def backward_hook(module, grad_input, grad_output):
        gradients['value'] = grad_output[0]

    fwd = target_layer.register_forward_hook(forward_hook)
    bwd = target_layer.register_full_backward_hook(backward_hook)

    input_tensor = image_tensor.unsqueeze(0).to(device)
    input_tensor.requires_grad = True

    output = net(input_tensor)
    if class_idx is None:
        class_idx = output.argmax(dim=1).item()

    net.zero_grad()
    output[0, class_idx].backward()
    fwd.remove(); bwd.remove()

    acts = activations['value'].squeeze(0)
    grads = gradients['value'].squeeze(0)
    weights = grads.mean(dim=(1, 2))

    cam = torch.zeros(acts.shape[1:]).to(device)
    for i, w in enumerate(weights):
        cam += w * acts[i]
    cam = F.relu(cam)
    cam = (cam - cam.min()) / (cam.max() + 1e-8)
    return cam.cpu().detach().numpy(), class_idx

def show_grad_cam(net, dataset, idx, classes, target_layer):
    image, true_label = dataset[idx]
    cam, predicted = grad_cam(net, image, target_layer)

    cam_t = torch.tensor(cam).unsqueeze(0).unsqueeze(0)
    cam_resized = F.interpolate(cam_t, size=image.shape[1:], mode='bilinear', align_corners=False)
    cam_resized = cam_resized.squeeze().numpy()

    img = (image.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img)
    axes[0].set_title(f"True: {classes[true_label]}")
    axes[0].axis('off')

    axes[1].imshow(img)
    axes[1].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[1].set_title(f"Predicted: {classes[predicted]}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

# net.features[-1] = last block (EfficientNetB0). Use net.conv2 if on the from-scratch CNN.
for i in random.sample(range(len(test_dataset)), 3):
    show_grad_cam(net, test_dataset, i, classes, net.features[-1])


# %% ================================================================
# 21. TEST-TIME AUGMENTATION (TTA)
# ====================================================================
def predict_with_tta(net, image, n_augments=5):
    net.eval()
    augment = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
    ])
    probs_sum = 0
    with torch.no_grad():
        for _ in range(n_augments):
            aug_img = augment(image).unsqueeze(0).to(device)
            probs_sum += F.softmax(net(aug_img), dim=1)
    return probs_sum / n_augments

image, true_label = test_dataset[0]
probs = predict_with_tta(net, image)
predicted = probs.argmax(dim=1).item()
print(f"True: {classes[true_label]}, TTA prediction: {classes[predicted]}")


# %% ================================================================
# 22. ENSEMBLING (inactive — requires 2+ separately trained + saved models)
# To use: train this script twice with different model options in section 6,
# saving each under a different checkpoint_path (e.g. 'best_model_A.pth',
# 'best_model_B.pth'), then delete the triple quotes below.
# ====================================================================
"""
net_a = efficientnet_b0(weights=None)
net_a.classifier[1] = nn.Linear(net_a.classifier[1].in_features, num_classes)
net_a.load_state_dict(torch.load('best_model_A.pth'))
net_a = net_a.to(device)

net_b = resnet50(weights=None)
net_b.fc = nn.Linear(net_b.fc.in_features, num_classes)
net_b.load_state_dict(torch.load('best_model_B.pth'))
net_b = net_b.to(device)

nets = [net_a, net_b]

def ensemble_predict(nets, image):
    probs_sum = 0
    with torch.no_grad():
        for m in nets:
            m.eval()
            probs_sum += F.softmax(m(image.unsqueeze(0).to(device)), dim=1)
    return probs_sum / len(nets)

image, true_label = test_dataset[0]
probs = ensemble_predict(nets, image)
predicted = probs.argmax(dim=1).item()
print(f"True: {classes[true_label]}, Ensemble prediction: {classes[predicted]}")
"""

Dataset already extracted — skipping unzip.
Using device: mps
Classes found: 100
Train samples: 13492
Valid samples: 500
Test samples:  500
Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
Label range: 1 to 97
EfficientNet loaded with 128100 trainable params
Output shape: torch.Size([32, 100])
Forward pass OK — shapes line up correctly.
[1,    84] loss: 3.990
[1,   168] loss: 2.978
[1,   252] loss: 2.462
[1,   336] loss: 2.225
[1,   420] loss: 2.039
Train accuracy: 80.55%
>>> Epoch 1 validation accuracy: 81.60%
  New best model saved (81.60%)
[2,    84] loss: 1.847
[2,   168] loss: 1.774
[2,   252] loss: 1.760
[2,   336] loss: 1.717
[2,   420] loss: 1.710
Train accuracy: 85.94%
>>> Epoch 2 validation accuracy: 87.20%
  New best model saved (87.20%)
[3,    84] loss: 1.604
[3,   168] loss: 1.582
[3,   252] loss: 1.599
[3,   336] loss: 1.587
[3,   420] loss: 1.601
Train accuracy: 88.25%
>>> Epoch 3 validation accuracy: 89.40%
  New best model saved (89.